# Contribution A Multi-Round Active Learning Pilot

This notebook runs the first real three-round M-OWODB Task-1 to Task-2
feedback-driven active annotation experiment for Contribution A using the
actual DAOWOD package and the actual PROB bridge.

Each variant starts from the same Task-1 checkpoint, controlled long-tail
candidate pool, base Task-1 reference set, empty Task-2 labelled set, fixed
evaluation split, seed, per-round budget, acquisition weights, training
configuration, and evaluation configuration. After round 1, each variant keeps
its own checkpoint, remaining pool, cumulative labelled set, and growing
novelty reference state.

Completed rounds are copied to Drive immediately after validation and can be
restored by a later runtime when their compact configuration metadata still
matches this notebook. This pilot is not a final benchmark result and should
not be read as automatic evidence that one variant is superior.


In [ ]:
# User-editable configuration.

SEED = 0
ROUNDS = 3
BUDGET_PER_ROUND = 10

SOURCE_TASK2_IMAGES = 300
REFERENCE_IMAGES = 30
EVAL_UNKNOWN_IMAGES = 100
EVAL_KNOWN_IMAGES = 100
IMBALANCE_RATIO = 20.0

TRAIN_EPOCHS = 1
BATCH_SIZE = 1
NUM_WORKERS = 2

ALPHA = 0.3
BETA = 0.2
GAMMA = 0.5
RARITY_POWER = 1.0
TOP_K = 3

VARIANTS = {
    "random": {
        "strategy": "random",
        "coherence_power": 1.0,
    },
    "rarity_no_coherence": {
        "strategy": "rarity_no_coherence",
        "coherence_power": 1.0,
    },
    "full_p05": {
        "strategy": "full",
        "coherence_power": 0.5,
    },
    "full_p1": {
        "strategy": "full",
        "coherence_power": 1.0,
    },
}

ALLOW_OVERWRITE_DRIVE_RESULTS = False
RESUME_COMPLETED_ROUNDS = True
RUN_PREFLIGHT_ONLY = False


# Repositories

DAOWOD_REPOSITORY_URL = (
    "https://github.com/gubiczam/"
    "distribution-aware-owod.git"
)

DAOWOD_COMMIT = "3f2763b2e54dd44b1cde27df5e2d8d87e54bf9e6"

PROB_REPOSITORY_URL = (
    "https://github.com/gubiczam/PROB.git"
)

PROB_BRANCH = "feat/daowod-bridge"


# Google Drive paths

DRIVE_ARCHIVE = (
    "/content/drive/MyDrive/DAOWOD/assets/OWOD_full.tar.zst"
)

DRIVE_TASK1_CHECKPOINT = (
    "/content/drive/MyDrive/DAOWOD/checkpoints/MOWODB/t1.pth"
)

EXPERIMENT_NAME = f"contribution_a_multiround_seed{SEED}"

DRIVE_RESULT_DIR = (
    f"/content/drive/MyDrive/DAOWOD/results/"
    f"{EXPERIMENT_NAME}"
)


# Local Colab paths

CONTENT_ROOT = "/content"

DAOWOD_PATH = (
    f"{CONTENT_ROOT}/distribution-aware-owod"
)

PROB_PATH = (
    f"{CONTENT_ROOT}/PROB"
)

DATA_ROOT = (
    f"{CONTENT_ROOT}/data/OWOD"
)

LOCAL_RESULT_ROOT = (
    f"/content/{EXPERIMENT_NAME}"
)

LOCAL_CHECKPOINT_DIR = (
    f"{CONTENT_ROOT}/daowod_checkpoints"
)

TASK1_CHECKPOINT_FOR_PROB = (
    f"{LOCAL_CHECKPOINT_DIR}/t1_epoch40.pth"
)

CANDIDATE_ANNOTATION_STASH = (
    f"{CONTENT_ROOT}/daowod_hidden_candidate_annotations"
)


# Dataset and evaluation configuration

DATASET = "TOWOD"

TASK2_SOURCE_SPLIT = "owod_t2_train"

TASK1_REFERENCE_SPLIT = "owod_t1_train"

OFFICIAL_EVAL_SPLIT = "owod_all_task_test"

PILOT_EVAL_SPLIT = (
    f"daowod_multiround_balanced_seed{SEED}_test"
)

PREVIOUS_CLASSES = 20
CURRENT_CLASSES = 20
NUM_CLASSES = 81

MAX_PROPOSALS_PER_IMAGE = 20
MINIMUM_UNKNOWN_SCORE = 0.0

DEVICE = "cuda"


# Pipeline status tracking

STATUS_ORDER = (
    "GPU",
    "Drive assets",
    "system packages",
    "repositories",
    "DAOWOD install",
    "DAOWOD validation",
    "PROB bridge",
    "attention backend",
    "protocol construction",
    "evaluation split",
    "selective extraction",
    "preflight",
    *(
        f"{variant} round {round_index}"
        for variant in VARIANTS
        for round_index in range(1, ROUNDS + 1)
    ),
    "final summary",
    "Drive persistence",
)

ROUND_STAGES = {
    f"{variant} round {round_index}"
    for variant in VARIANTS
    for round_index in range(1, ROUNDS + 1)
}

STATUS = dict.fromkeys(
    STATUS_ORDER,
    "PENDING",
)

EXPECTED_VARIANTS = {
    "random": {"strategy": "random", "coherence_power": 1.0},
    "rarity_no_coherence": {
        "strategy": "rarity_no_coherence",
        "coherence_power": 1.0,
    },
    "full_p05": {"strategy": "full", "coherence_power": 0.5},
    "full_p1": {"strategy": "full", "coherence_power": 1.0},
}
if VARIANTS != EXPECTED_VARIANTS:
    raise RuntimeError("The four configured variants differ from the experiment protocol.")


In [ ]:
import gc
import hashlib
import json
import platform
import random
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd


def run_cmd(command, *, cwd=None, timeout=600, check=True):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
        check=False,
    )
    if result.stdout:
        print(result.stdout[-8000:])
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command, output=result.stdout)
    return result


def set_status(stage, value="OK"):
    STATUS[stage] = value
    print(f"{stage}: {value}")


def mark(stage):
    set_status(stage, "OK")


def mark_restored(stage):
    set_status(stage, "RESTORED")


def read_ids(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing image set: {path}")
    return [line.split()[0] for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def write_ids(path, image_ids):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(image_ids) + ("\n" if image_ids else ""), encoding="utf-8")


def unique_preserve(values):
    return list(dict.fromkeys(str(value) for value in values))


def disk_free_gb(path="/content"):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return round(shutil.disk_usage(path).free / 1024**3, 2)


def sha256_text(values):
    digest = hashlib.sha256()
    for value in values:
        digest.update(str(value).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


def json_default(value):
    if hasattr(value, "item"):
        return value.item()
    if hasattr(value, "tolist"):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    temporary.write_text(
        json.dumps(data, indent=2, sort_keys=True, default=json_default) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def cleanup_runtime():
    gc.collect()
    if "torch" in globals() and torch.cuda.is_available():
        torch.cuda.empty_cache()


import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. Select a Colab T4 GPU runtime.")

drive.mount("/content/drive", force_remount=False)
archive = Path(DRIVE_ARCHIVE)
task1_checkpoint = Path(DRIVE_TASK1_CHECKPOINT)
missing_drive_assets = [str(path) for path in (archive, task1_checkpoint) if not path.exists()]
if missing_drive_assets:
    raise FileNotFoundError("Missing Drive asset(s): " + ", ".join(missing_drive_assets))

Path(LOCAL_RESULT_ROOT).mkdir(parents=True, exist_ok=True)
Path(DRIVE_RESULT_DIR).mkdir(parents=True, exist_ok=True)
if not Path(DRIVE_RESULT_DIR).is_dir():
    raise RuntimeError(f"Drive result path is not a directory: {DRIVE_RESULT_DIR}")

runtime_rows = {
    "Python": sys.version.split()[0],
    "Platform": platform.platform(),
    "PyTorch": torch.__version__,
    "CUDA": torch.version.cuda,
    "GPU": torch.cuda.get_device_name(0),
    "Archive": str(archive),
    "Task-1 checkpoint": str(task1_checkpoint),
    "Expected training runs": len(VARIANTS) * ROUNDS,
    "Local output": LOCAL_RESULT_ROOT,
    "Drive output": DRIVE_RESULT_DIR,
    "Free /content GB": disk_free_gb("/content"),
    "Free Drive GB": disk_free_gb(Path(DRIVE_RESULT_DIR).parent),
}
display(pd.DataFrame(runtime_rows.items(), columns=["item", "value"]))
mark("GPU")
mark("Drive assets")


In [ ]:
run_cmd(["apt-get", "update", "-qq"], timeout=600)
run_cmd(["apt-get", "install", "-y", "-qq", "zstd", "ninja-build", "build-essential"], timeout=900)
zstd_version = run_cmd(["zstd", "--version"], timeout=60).stdout.strip()
ninja_version = run_cmd(["ninja", "--version"], timeout=60).stdout.strip()
display(pd.DataFrame({"package": ["zstd", "ninja"], "version": [zstd_version, ninja_version]}))
mark("system packages")


In [ ]:
for checkout in (Path(DAOWOD_PATH), Path(PROB_PATH), Path(LOCAL_CHECKPOINT_DIR)):
    if checkout.exists():
        shutil.rmtree(checkout)

run_cmd(["git", "clone", DAOWOD_REPOSITORY_URL, DAOWOD_PATH], cwd=CONTENT_ROOT, timeout=300)
run_cmd(["git", "checkout", DAOWOD_COMMIT], cwd=DAOWOD_PATH, timeout=120)
run_cmd(
    [
        "git",
        "clone",
        "--branch",
        PROB_BRANCH,
        "--single-branch",
        PROB_REPOSITORY_URL,
        PROB_PATH,
    ],
    cwd=CONTENT_ROOT,
    timeout=300,
)
repo_commits = {
    "DAOWOD": run_cmd(["git", "rev-parse", "HEAD"], cwd=DAOWOD_PATH).stdout.strip(),
    "PROB": run_cmd(["git", "rev-parse", "HEAD"], cwd=PROB_PATH).stdout.strip(),
}
if repo_commits["DAOWOD"] != DAOWOD_COMMIT:
    raise RuntimeError(
        "DAOWOD checkout mismatch: "
        f"expected {DAOWOD_COMMIT}, got {repo_commits['DAOWOD']}"
    )
display(pd.DataFrame(repo_commits.items(), columns=["repository", "commit"]))
mark("repositories")


In [ ]:
import importlib
import inspect

run_cmd([sys.executable, "-m", "pip", "install", "--editable", f"{DAOWOD_PATH}[dev]"], timeout=900)
daowod_src = str(Path(DAOWOD_PATH) / "src")
if daowod_src not in sys.path:
    sys.path.insert(0, daowod_src)
importlib.invalidate_caches()

import daowod
from daowod import ProbAdapter, load_config, run_active_round
from daowod.acquisition import AcquisitionWeights
from daowod.config import AcquisitionConfig
from daowod.dataset import build_long_tail_pool

imported_path = Path(daowod.__file__).resolve()
expected_root = (Path(DAOWOD_PATH) / "src" / "daowod").resolve()
print("daowod.__file__ =", imported_path)
if expected_root not in imported_path.parents and imported_path != expected_root / "__init__.py":
    raise RuntimeError(f"Imported daowod from unexpected location: {imported_path}")

round_signature = inspect.signature(run_active_round)
required_round_parameters = {
    "adapter",
    "checkpoint",
    "candidate_ids",
    "reference_ids",
    "labelled_ids",
    "output_dir",
    "strategy",
    "budget",
    "acquisition_config",
    "seed",
    "round_index",
}
missing_round_parameters = required_round_parameters - set(round_signature.parameters)
if missing_round_parameters:
    raise RuntimeError(f"run_active_round is missing required parameters: {sorted(missing_round_parameters)}")
if not callable(run_active_round):
    raise RuntimeError("run_active_round is not callable.")
print("run_active_round signature:", round_signature)

base_config = load_config(Path(DAOWOD_PATH) / "configs" / "experiment.yaml")
base_acquisition = base_config.acquisition


def build_acquisition_config(variant_config):
    strategy = variant_config["strategy"]
    config_strategy = "random" if strategy == "random" else "full"
    weights = AcquisitionWeights(
        uncertainty=ALPHA,
        novelty=BETA,
        rarity=GAMMA,
        coherence_power=float(variant_config["coherence_power"]),
        rarity_power=RARITY_POWER,
    )
    return AcquisitionConfig(
        strategies=(config_strategy,),
        uncertainty_mode=base_acquisition.uncertainty_mode,
        pseudo_label_source=base_acquisition.pseudo_label_source,
        cluster_count=base_acquisition.cluster_count,
        neighbour_count=base_acquisition.neighbour_count,
        top_k=TOP_K,
        weights=weights,
    )


variant_rows = []
for variant_name, variant_config in VARIANTS.items():
    config = build_acquisition_config(variant_config)
    variant_rows.append(
        {
            "variant": variant_name,
            "underlying_strategy": variant_config["strategy"],
            "coherence_power": config.weights.coherence_power,
            "uncertainty_mode": config.uncertainty_mode,
            "pseudo_label_source": config.pseudo_label_source,
            "cluster_count": config.cluster_count,
            "neighbour_count": config.neighbour_count,
            "top_k": config.top_k,
            "alpha": config.weights.uncertainty,
            "beta": config.weights.novelty,
            "gamma": config.weights.rarity,
            "rarity_power": config.weights.rarity_power,
        }
    )
effective_acquisition_parameters = variant_rows
display(pd.DataFrame(variant_rows))
mark("DAOWOD install")


In [ ]:
import importlib.util

prob_dependencies = {
    "wandb": "wandb",
    "einops": "einops",
    "pycocotools": "pycocotools",
    "skimage": "scikit-image",
    "joblib": "joblib",
    "tqdm": "tqdm",
    "ipdb": "ipdb",
}
missing_packages = [package for module, package in prob_dependencies.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    run_cmd([sys.executable, "-m", "pip", "install", *missing_packages], cwd=PROB_PATH, timeout=900)
else:
    print("PROB dependency imports already available.")

run_cmd([sys.executable, "-m", "ruff", "check", "."], cwd=DAOWOD_PATH, timeout=300)
run_cmd([sys.executable, "-m", "pytest", "-q"], cwd=DAOWOD_PATH, timeout=300)
run_cmd([sys.executable, "-m", "compileall", "-q", "src", "tests"], cwd=DAOWOD_PATH, timeout=300)
mark("DAOWOD validation")
run_cmd([sys.executable, "daowod_prob_bridge.py", "check"], cwd=PROB_PATH, timeout=120)
mark("PROB bridge")


In [ ]:
def require_text(path, needles, description):
    source = Path(path).read_text(encoding="utf-8")
    if not any(needle in source for needle in needles):
        raise RuntimeError(f"Missing {description}: {path}")
    compile(source, str(path), "exec")
    return source


def enable_attention_fallback():
    func_path = Path(PROB_PATH) / "models" / "ops" / "functions" / "ms_deform_attn_func.py"
    module_path = Path(PROB_PATH) / "models" / "ops" / "modules" / "ms_deform_attn.py"

    func_source = func_path.read_text(encoding="utf-8")
    original_import = "import MultiScaleDeformableAttention as MSDA"
    fallback_import = "try:\n    import MultiScaleDeformableAttention as MSDA\nexcept ModuleNotFoundError:\n    MSDA = None"
    if fallback_import in func_source:
        pass
    elif original_import in func_source:
        func_source = func_source.replace(original_import, fallback_import)
        func_path.write_text(func_source, encoding="utf-8")
    else:
        raise RuntimeError(f"Unknown attention function import form: {func_path}")

    module_source = module_path.read_text(encoding="utf-8")
    original_module_import = "from ..functions import MSDeformAttnFunction"
    fallback_module_import = "from ..functions.ms_deform_attn_func import MSDeformAttnFunction, ms_deform_attn_core_pytorch"
    if fallback_module_import in module_source:
        pass
    elif original_module_import in module_source:
        module_source = module_source.replace(original_module_import, fallback_module_import)
    else:
        raise RuntimeError(f"Unknown attention module import form: {module_path}")

    original_call = (
        "output = MSDeformAttnFunction.apply(\n"
        "            value, input_spatial_shapes, input_level_start_index, sampling_locations, attention_weights, self.im2col_step)"
    )
    fallback_call = (
        "output = ms_deform_attn_core_pytorch(\n"
        "            value, input_spatial_shapes, sampling_locations, attention_weights)"
    )
    if fallback_call in module_source:
        pass
    elif original_call in module_source:
        module_source = module_source.replace(original_call, fallback_call)
    else:
        raise RuntimeError(f"Unknown attention forward form: {module_path}")
    module_path.write_text(module_source, encoding="utf-8")
    print("Enabled pure-PyTorch deformable-attention fallback.")


def attention_forward_backward_check():
    code = "\n".join([
        "import torch",
        "from models.ops.modules.ms_deform_attn import MSDeformAttn",
        "assert torch.cuda.is_available()",
        "device = torch.device('cuda')",
        "module = MSDeformAttn(d_model=8, n_levels=1, n_heads=2, n_points=2).to(device)",
        "query = torch.randn(1, 3, 8, device=device, requires_grad=True)",
        "reference_points = torch.full((1, 3, 1, 2), 0.5, device=device)",
        "input_flatten = torch.randn(1, 4, 8, device=device, requires_grad=True)",
        "input_spatial_shapes = torch.tensor([[2, 2]], dtype=torch.long, device=device)",
        "input_level_start_index = torch.tensor([0], dtype=torch.long, device=device)",
        "output = module(query, reference_points, input_flatten, input_spatial_shapes, input_level_start_index)",
        "loss = output.square().mean()",
        "loss.backward()",
        "assert output.shape == (1, 3, 8)",
        "assert query.grad is not None and torch.isfinite(query.grad).all()",
        "assert input_flatten.grad is not None and torch.isfinite(input_flatten.grad).all()",
        "print('attention forward/backward OK', tuple(output.shape))",
    ])
    run_cmd([sys.executable, "-c", code], cwd=PROB_PATH, timeout=180)


require_text(Path(PROB_PATH) / "models" / "prob_deformable_detr.py", ["'pred_features': hs[-1]", '"pred_features": hs[-1]'], "decoder-feature export patch")
require_text(Path(PROB_PATH) / "main_open_world.py", ["test_stats = {}"], "main_open_world.py test_stats initialization fix")

compile_result = run_cmd(["bash", "make.sh"], cwd=Path(PROB_PATH) / "models" / "ops", timeout=900, check=False)
attention_backend = "CUDA extension"
if compile_result.returncode != 0:
    attention_backend = "pure PyTorch fallback"
    enable_attention_fallback()

attention_forward_backward_check()

backbone_path = Path(PROB_PATH) / "models" / "dino_resnet50_pretrain.pth"
if not backbone_path.exists():
    run_cmd([sys.executable, "-c", "import urllib.request; urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth', 'models/dino_resnet50_pretrain.pth')"], cwd=PROB_PATH, timeout=1800)
if not backbone_path.exists():
    raise FileNotFoundError(f"Missing DINO backbone: {backbone_path}")

Path(LOCAL_CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
checkpoint = torch.load(DRIVE_TASK1_CHECKPOINT, map_location="cpu", weights_only=False)
if isinstance(checkpoint, dict) and isinstance(checkpoint.get("epoch"), int):
    shutil.copy2(DRIVE_TASK1_CHECKPOINT, TASK1_CHECKPOINT_FOR_PROB)
    task1_epoch = checkpoint["epoch"]
elif isinstance(checkpoint, dict):
    wrapped = dict(checkpoint)
    wrapped["epoch"] = 40
    torch.save(wrapped, TASK1_CHECKPOINT_FOR_PROB)
    task1_epoch = 40
else:
    torch.save({"model": checkpoint, "epoch": 40}, TASK1_CHECKPOINT_FOR_PROB)
    task1_epoch = 40
verified = torch.load(TASK1_CHECKPOINT_FOR_PROB, map_location="cpu", weights_only=False)
if not isinstance(verified, dict) or not isinstance(verified.get("epoch"), int):
    raise RuntimeError("Task-1 checkpoint wrapper did not produce an integer epoch.")

display(pd.DataFrame({"item": ["attention backend", "Task-1 checkpoint", "Task-1 epoch", "DINO backbone"], "value": [attention_backend, TASK1_CHECKPOINT_FOR_PROB, task1_epoch, str(backbone_path)]}))
mark("attention backend")


In [ ]:
data_parent = Path(DATA_ROOT).parent
data_parent.mkdir(parents=True, exist_ok=True)
if Path(DATA_ROOT).exists():
    shutil.rmtree(DATA_ROOT)

run_cmd([
    "bash",
    "-lc",
    "zstd -dc \"$1\" | tar -xf - -C \"$2\" OWOD/ImageSets OWOD/Annotations",
    "bash",
    DRIVE_ARCHIVE,
    str(data_parent),
], timeout=1800)
required_paths = [
    Path(DATA_ROOT) / "ImageSets" / DATASET / f"{TASK2_SOURCE_SPLIT}.txt",
    Path(DATA_ROOT) / "ImageSets" / DATASET / f"{TASK1_REFERENCE_SPLIT}.txt",
    Path(DATA_ROOT) / "ImageSets" / DATASET / f"{OFFICIAL_EVAL_SPLIT}.txt",
    Path(DATA_ROOT) / "Annotations",
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Archive did not provide required protocol files: " + ", ".join(missing_paths))
annotation_count = sum(1 for _ in (Path(DATA_ROOT) / "Annotations").glob("*.xml"))
split_counts = {
    TASK2_SOURCE_SPLIT: len(read_ids(required_paths[0])),
    TASK1_REFERENCE_SPLIT: len(read_ids(required_paths[1])),
    OFFICIAL_EVAL_SPLIT: len(read_ids(required_paths[2])),
}
display(pd.DataFrame({"item": ["annotations", *split_counts], "count": [annotation_count, *split_counts.values()]}))


In [ ]:
import contextlib
import io
import xml.etree.ElementTree as ET

prob_src = str(Path(PROB_PATH).resolve())
if prob_src not in sys.path:
    sys.path.insert(0, prob_src)
with contextlib.redirect_stdout(io.StringIO()):
    from datasets.torchvision_datasets.open_world import (  # noqa: E402
        BASE_VOC_CLASS_NAMES,
        VOC_CLASS_NAMES_COCOFIED,
        VOC_COCO_CLASS_NAMES,
    )


def seeded_order_preserving_sample(image_ids, count, *, seed, salt):
    unique_ids = list(dict.fromkeys(str(image_id) for image_id in image_ids))
    if len(unique_ids) != len(image_ids):
        raise ValueError(f"{salt} source IDs contain duplicates.")
    if count > len(unique_ids):
        raise ValueError(f"Requested {count} images from only {len(unique_ids)} IDs for {salt}.")
    rng = random.Random(f"{seed}:{salt}")
    selected = set(rng.sample(unique_ids, count))
    return [image_id for image_id in unique_ids if image_id in selected]


image_set_root = Path(DATA_ROOT) / "ImageSets" / DATASET
official_task2_ids = read_ids(image_set_root / f"{TASK2_SOURCE_SPLIT}.txt")
task2_source_ids = seeded_order_preserving_sample(
    official_task2_ids,
    SOURCE_TASK2_IMAGES,
    seed=SEED,
    salt="task2-source",
)
pilot_source_split = image_set_root / "daowod_multiround_t2_source_train.txt"
write_ids(pilot_source_split, task2_source_ids)

task_class_names = list(
    VOC_COCO_CLASS_NAMES[DATASET][PREVIOUS_CLASSES : PREVIOUS_CLASSES + CURRENT_CLASSES]
)
long_tail_dir = Path(LOCAL_RESULT_ROOT) / "protocol" / "long_tail"
pool = build_long_tail_pool(
    annotation_dir=Path(DATA_ROOT) / "Annotations",
    source_split=pilot_source_split,
    task_class_names=task_class_names,
    output_dir=long_tail_dir,
    imbalance_ratio=IMBALANCE_RATIO,
    seed=SEED,
)
candidate_ids = [str(image_id) for image_id in pool["selected_image_ids"]]
official_t1_ids = read_ids(image_set_root / f"{TASK1_REFERENCE_SPLIT}.txt")
reference_source_ids = [image_id for image_id in official_t1_ids if image_id not in set(candidate_ids)]
base_reference_ids = seeded_order_preserving_sample(
    reference_source_ids,
    REFERENCE_IMAGES,
    seed=SEED,
    salt="reference",
)
initial_labelled_ids = []

if len(candidate_ids) != len(set(candidate_ids)):
    raise RuntimeError("Candidate IDs are not unique.")
if len(base_reference_ids) != len(set(base_reference_ids)):
    raise RuntimeError("Base reference IDs are not unique.")
if set(candidate_ids) & set(base_reference_ids):
    raise RuntimeError("Candidate and base reference IDs overlap.")
if len(candidate_ids) < ROUNDS * BUDGET_PER_ROUND:
    raise RuntimeError(
        f"Candidate pool has {len(candidate_ids)} images, "
        f"below cumulative budget {ROUNDS * BUDGET_PER_ROUND}."
    )
for artifact in (pool["pool_split_path"], pool["class_stats_path"], pool["manifest_path"]):
    if not Path(artifact).exists():
        raise FileNotFoundError(f"Missing protocol artifact: {artifact}")

protocol_dir = Path(LOCAL_RESULT_ROOT) / "protocol"
write_ids(protocol_dir / "initial_candidate_ids.txt", candidate_ids)
write_ids(protocol_dir / "base_reference_ids.txt", base_reference_ids)
write_ids(protocol_dir / "initial_labelled_ids.txt", initial_labelled_ids)

training_protocol = {
    "seed": SEED,
    "rounds": ROUNDS,
    "budget_per_round": BUDGET_PER_ROUND,
    "source_split": str(pilot_source_split),
    "source_image_count": len(task2_source_ids),
    "candidate_pool_count": len(candidate_ids),
    "base_reference_count": len(base_reference_ids),
    "initial_labelled_count": len(initial_labelled_ids),
    "imbalance_ratio": IMBALANCE_RATIO,
    "task_class_names": task_class_names,
    "variants": VARIANTS,
    "candidate_ground_truth_use": "controlled long-tail protocol construction only",
    "candidate_ground_truth_visible_to_acquisition": False,
    "reference_growth": "base Task-1 references plus all Task-2 images labelled before the round",
}
training_protocol_path = protocol_dir / "training_protocol.json"
write_json(training_protocol_path, training_protocol)
display(pd.DataFrame(training_protocol.items(), columns=["item", "value"]))
mark("protocol construction")


In [ ]:
def annotation_classes(image_id):
    path = Path(DATA_ROOT) / "Annotations" / f"{image_id}.xml"
    if not path.exists():
        raise FileNotFoundError(f"Missing annotation: {path}")
    root = ET.parse(path).getroot()
    classes = []
    for node in root.findall("./object/name"):
        if node.text:
            name = node.text.strip()
            if name in VOC_CLASS_NAMES_COCOFIED:
                name = BASE_VOC_CLASS_NAMES[VOC_CLASS_NAMES_COCOFIED.index(name)]
            classes.append(name)
    return classes


class_names = list(VOC_COCO_CLASS_NAMES[DATASET])
class_to_index = {name: index for index, name in enumerate(class_names)}
official_eval_ids_raw = read_ids(image_set_root / f"{OFFICIAL_EVAL_SPLIT}.txt")
official_eval_ids = list(dict.fromkeys(official_eval_ids_raw))
official_eval_duplicate_count = len(official_eval_ids_raw) - len(official_eval_ids)
unknown_candidates = []
known_only_candidates = []
unknown_object_count = 0
for image_id in official_eval_ids:
    indices = [class_to_index[name] for name in annotation_classes(image_id) if name in class_to_index]
    if not indices:
        continue
    has_unknown = any(40 <= index <= 79 for index in indices)
    known_only = all(0 <= index <= 39 for index in indices)
    if has_unknown:
        unknown_candidates.append(image_id)
        unknown_object_count += sum(40 <= index <= 79 for index in indices)
    elif known_only:
        known_only_candidates.append(image_id)

unknown_eval_ids = seeded_order_preserving_sample(
    unknown_candidates,
    EVAL_UNKNOWN_IMAGES,
    seed=SEED,
    salt="eval-unknown",
)
known_eval_ids = seeded_order_preserving_sample(
    known_only_candidates,
    EVAL_KNOWN_IMAGES,
    seed=SEED,
    salt="eval-known",
)
evaluation_selection = set(unknown_eval_ids) | set(known_eval_ids)
evaluation_ids = [image_id for image_id in official_eval_ids if image_id in evaluation_selection]
evaluation_ids_sha256 = sha256_text(evaluation_ids)
evaluation_split_path = image_set_root / f"{PILOT_EVAL_SPLIT}.txt"
write_ids(evaluation_split_path, evaluation_ids)
write_ids(protocol_dir / "evaluation_ids.txt", evaluation_ids)

if "test" not in PILOT_EVAL_SPLIT:
    raise RuntimeError("Evaluation split name must contain 'test'.")
if len(evaluation_ids) != EVAL_UNKNOWN_IMAGES + EVAL_KNOWN_IMAGES:
    raise RuntimeError(f"Expected 200 evaluation IDs, got {len(evaluation_ids)}.")
if len(evaluation_ids) != len(set(evaluation_ids)):
    raise RuntimeError("Evaluation IDs contain duplicates.")
if set(candidate_ids) & set(evaluation_ids):
    raise RuntimeError("Candidate IDs overlap the fixed evaluation split.")
if unknown_object_count < 1:
    raise RuntimeError("Evaluation candidates contain no unknown ground-truth objects.")
for image_id in evaluation_ids:
    if not (Path(DATA_ROOT) / "Annotations" / f"{image_id}.xml").exists():
        raise FileNotFoundError(f"Missing evaluation XML: {image_id}")

evaluation_protocol = {
    "seed": SEED,
    "official_source_split": OFFICIAL_EVAL_SPLIT,
    "official_source_raw_count": len(official_eval_ids_raw),
    "official_source_unique_count": len(official_eval_ids),
    "official_source_duplicate_count": official_eval_duplicate_count,
    "pilot_split": PILOT_EVAL_SPLIT,
    "known_class_indices": "0..39",
    "unknown_class_indices": "40..79",
    "selected_unknown_count": len(unknown_eval_ids),
    "selected_known_count": len(known_eval_ids),
    "evaluation_count": len(evaluation_ids),
    "evaluation_ids_sha256": evaluation_ids_sha256,
    "unknown_ground_truth_objects_in_source_candidates": unknown_object_count,
    "ground_truth_use": "evaluation_subset_construction_and_metrics_only",
}
evaluation_protocol_path = protocol_dir / "evaluation_protocol.json"
write_json(evaluation_protocol_path, evaluation_protocol)
display(pd.DataFrame(evaluation_protocol.items(), columns=["item", "value"]))
mark("evaluation split")


In [ ]:
required_image_ids = list(dict.fromkeys([*candidate_ids, *base_reference_ids, *evaluation_ids]))
member_list_path = Path(CONTENT_ROOT) / "daowod_required_jpegs.txt"
member_list_path.write_text(
    "\n".join(f"OWOD/JPEGImages/{image_id}.jpg" for image_id in required_image_ids) + "\n",
    encoding="utf-8",
)
run_cmd([
    "bash",
    "-lc",
    "zstd -dc \"$1\" | tar -xf - -C \"$2\" -T \"$3\"",
    "bash",
    DRIVE_ARCHIVE,
    str(Path(DATA_ROOT).parent),
    str(member_list_path),
], timeout=3600)
missing_required = []
for image_id in required_image_ids:
    jpeg_path = Path(DATA_ROOT) / "JPEGImages" / f"{image_id}.jpg"
    if not jpeg_path.exists():
        missing_required.append(str(jpeg_path))
for image_id in [*base_reference_ids, *evaluation_ids]:
    annotation_path = Path(DATA_ROOT) / "Annotations" / f"{image_id}.xml"
    if not annotation_path.exists():
        missing_required.append(str(annotation_path))
if missing_required:
    raise FileNotFoundError("Selective extraction missing required files: " + ", ".join(missing_required[:20]))

annotation_stash = Path(CANDIDATE_ANNOTATION_STASH)
if annotation_stash.exists():
    shutil.rmtree(annotation_stash)
annotation_stash.mkdir(parents=True, exist_ok=True)
visible_annotation_ids = set(base_reference_ids) | set(evaluation_ids)
hidden_candidate_annotations = 0
for image_id in candidate_ids:
    if image_id in visible_annotation_ids:
        continue
    source = Path(DATA_ROOT) / "Annotations" / f"{image_id}.xml"
    if source.exists():
        shutil.move(str(source), annotation_stash / source.name)
        hidden_candidate_annotations += 1

jpeg_size_gb = round(
    sum(
        (Path(DATA_ROOT) / "JPEGImages" / f"{image_id}.jpg").stat().st_size
        for image_id in required_image_ids
    )
    / 1024**3,
    3,
)
extraction_rows = {
    "candidate count": len(candidate_ids),
    "base reference count": len(base_reference_ids),
    "evaluation count": len(evaluation_ids),
    "unique required JPEG count": len(required_image_ids),
    "hidden candidate annotation count": hidden_candidate_annotations,
    "extracted JPEG GB": jpeg_size_gb,
    "free /content GB": disk_free_gb("/content"),
}
display(pd.DataFrame(extraction_rows.items(), columns=["item", "value"]))
mark("selective extraction")


In [ ]:
python_command = sys.executable
common_prob_args = (
    f"--data-root {DATA_ROOT} --dataset {DATASET} "
    f"--prev-introduced-classes {PREVIOUS_CLASSES} --current-introduced-classes {CURRENT_CLASSES} "
    f"--num-classes {NUM_CLASSES} --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} "
    f"--device {DEVICE} --seed {SEED}"
)
adapter = ProbAdapter(
    repository_path=PROB_PATH,
    timeout_seconds=86400,
    train_command=(
        f"{python_command} daowod_prob_bridge.py train "
        "--labelled-ids {labelled_ids} --previous-checkpoint {previous_checkpoint} "
        "--output-checkpoint {checkpoint} --output-dir {output_dir} "
        f"--test-set {PILOT_EVAL_SPLIT} --epochs {TRAIN_EPOCHS} {common_prob_args}"
    ),
    predict_command=(
        f"{python_command} daowod_prob_bridge.py predict "
        "--image-ids {image_ids} --checkpoint {checkpoint} --output {proposals} "
        f"--max-proposals-per-image {MAX_PROPOSALS_PER_IMAGE} "
        f"--minimum-unknown-score {MINIMUM_UNKNOWN_SCORE} {common_prob_args}"
    ),
    evaluate_command=(
        f"{python_command} daowod_prob_bridge.py evaluate "
        "--checkpoint {checkpoint} --output {metrics} --output-dir {output_dir} "
        f"--test-set {PILOT_EVAL_SPLIT} {common_prob_args}"
    ),
)
original_adapter_train = adapter.train


def reveal_annotations_for_training(labelled_image_ids, *, previous_checkpoint, run_dir, round_index, seed):
    annotation_dir = Path(DATA_ROOT) / "Annotations"
    annotation_dir.mkdir(parents=True, exist_ok=True)
    for image_id in labelled_image_ids:
        visible = annotation_dir / f"{image_id}.xml"
        hidden = Path(CANDIDATE_ANNOTATION_STASH) / f"{image_id}.xml"
        if not visible.exists() and hidden.exists():
            shutil.copy2(hidden, visible)
    return original_adapter_train(
        labelled_image_ids,
        previous_checkpoint=previous_checkpoint,
        run_dir=run_dir,
        round_index=round_index,
        seed=seed,
    )


adapter.train = reveal_annotations_for_training

experiment_config = {
    "seed": int(SEED),
    "rounds": int(ROUNDS),
    "budget_per_round": int(BUDGET_PER_ROUND),
    "variants": VARIANTS,
    "acquisition_weights": {
        "uncertainty": float(ALPHA),
        "novelty": float(BETA),
        "rarity": float(GAMMA),
        "rarity_power": float(RARITY_POWER),
        "top_k": int(TOP_K),
        "uncertainty_mode": base_acquisition.uncertainty_mode,
        "pseudo_label_source": base_acquisition.pseudo_label_source,
        "cluster_count": int(base_acquisition.cluster_count),
        "neighbour_count": int(base_acquisition.neighbour_count),
    },
    "training_settings": {
        "epochs": int(TRAIN_EPOCHS),
        "batch_size": int(BATCH_SIZE),
        "num_workers": int(NUM_WORKERS),
        "max_proposals_per_image": int(MAX_PROPOSALS_PER_IMAGE),
        "minimum_unknown_score": float(MINIMUM_UNKNOWN_SCORE),
    },
    "dataset_protocol_settings": {
        "dataset": DATASET,
        "task2_source_split": TASK2_SOURCE_SPLIT,
        "task1_reference_split": TASK1_REFERENCE_SPLIT,
        "official_eval_split": OFFICIAL_EVAL_SPLIT,
        "pilot_eval_split": PILOT_EVAL_SPLIT,
        "source_task2_images": int(SOURCE_TASK2_IMAGES),
        "reference_images": int(REFERENCE_IMAGES),
        "eval_unknown_images": int(EVAL_UNKNOWN_IMAGES),
        "eval_known_images": int(EVAL_KNOWN_IMAGES),
        "imbalance_ratio": float(IMBALANCE_RATIO),
        "initial_candidate_count": int(len(candidate_ids)),
        "base_reference_count": int(len(base_reference_ids)),
        "evaluation_count": int(len(evaluation_ids)),
        "evaluation_ids_sha256": evaluation_ids_sha256,
    },
    "DAOWOD_commit": str(repo_commits["DAOWOD"]),
    "PROB_commit": str(repo_commits["PROB"]),
}
experiment_config_path = Path(LOCAL_RESULT_ROOT) / "experiment_config.json"
drive_config_path = Path(DRIVE_RESULT_DIR) / "experiment_config.json"
if drive_config_path.exists():
    saved_config = json.loads(drive_config_path.read_text(encoding="utf-8"))
    if saved_config != experiment_config:
        raise RuntimeError("Existing Drive experiment_config.json is incompatible with this notebook.")
elif any(Path(DRIVE_RESULT_DIR).iterdir()):
    raise RuntimeError(
        "Drive result directory exists without experiment_config.json. "
        "Refusing to mix results from another experiment."
    )
write_json(experiment_config_path, experiment_config)
write_json(drive_config_path, experiment_config)

preflight_rows = {
    "GPU availability": torch.cuda.is_available(),
    "GPU device": torch.cuda.get_device_name(0),
    "Drive mounted": Path("/content/drive").exists(),
    "archive exists": Path(DRIVE_ARCHIVE).exists(),
    "Task-1 checkpoint exists": Path(DRIVE_TASK1_CHECKPOINT).exists(),
    "repositories checked out": Path(DAOWOD_PATH).exists() and Path(PROB_PATH).exists(),
    "exact DAOWOD commit": repo_commits["DAOWOD"] == DAOWOD_COMMIT,
    "exact PROB commit": repo_commits["PROB"],
    "run_active_round callable": callable(run_active_round),
    "initial candidate count": len(candidate_ids),
    "base reference count": len(base_reference_ids),
    "evaluation count": len(evaluation_ids),
    "candidate/reference disjoint": set(candidate_ids).isdisjoint(base_reference_ids),
    "four variants": len(VARIANTS) == 4,
    "three rounds": ROUNDS == 3,
    "twelve expected training runs": len(VARIANTS) * ROUNDS,
    "local output path": LOCAL_RESULT_ROOT,
    "Drive output path": DRIVE_RESULT_DIR,
    "local free GB": disk_free_gb("/content"),
    "Drive free GB": disk_free_gb(Path(DRIVE_RESULT_DIR).parent),
}
display(pd.DataFrame(preflight_rows.items(), columns=["preflight_check", "value"]))
mark("preflight")
if RUN_PREFLIGHT_ONLY:
    raise SystemExit("RUN_PREFLIGHT_ONLY=True; stopping before the first active round.")


In [ ]:
import csv

import numpy as np

REQUIRED_METRICS = {
    "known_mAP",
    "U_Recall",
    "WI",
    "A_OSE",
    "unknown_AP50",
    "previous_known_AP50",
}


def round_dir(variant_name, round_index, *, root=LOCAL_RESULT_ROOT):
    return Path(root) / variant_name / f"round_{round_index:02d}"


def round_metadata_path(directory):
    return Path(directory) / "round_metadata.json"


def required_round_files(variant_name, directory):
    directory = Path(directory)
    files = [
        directory / "candidate_proposals.npz",
        directory / "selected_ids.txt",
        directory / "labelled_ids.txt",
        directory / "remaining_pool_ids.txt",
        directory / "checkpoint.pth",
        directory / "metrics.json",
        directory / "round_manifest.json",
    ]
    if VARIANTS[variant_name]["strategy"] != "random":
        files.extend(
            [
                directory / "reference_proposals.npz",
                directory / "proposal_scores.csv",
                directory / "image_scores.csv",
            ]
        )
    return files


def validate_round_artifacts(variant_name, round_index, directory, record, *, resumed):
    directory = Path(directory)
    missing = [str(path) for path in required_round_files(variant_name, directory) if not path.exists()]
    if missing:
        raise RuntimeError(f"Missing {variant_name} round {round_index} artifacts: {missing}")
    manifest = json.loads((directory / "round_manifest.json").read_text(encoding="utf-8"))
    metrics = json.loads((directory / "metrics.json").read_text(encoding="utf-8"))
    selected = read_ids(directory / "selected_ids.txt")
    labelled = read_ids(directory / "labelled_ids.txt")
    remaining = read_ids(directory / "remaining_pool_ids.txt")
    strategy = VARIANTS[variant_name]["strategy"]

    assert manifest["completed"] is True
    assert manifest["strategy"] == strategy
    assert manifest["round_index"] == round_index
    assert manifest["seed"] == SEED
    assert manifest["budget"] == BUDGET_PER_ROUND
    assert manifest["candidate_count_before"] == len(record["input_candidate_ids"])
    assert len(selected) == len(set(selected)) == BUDGET_PER_ROUND
    assert set(selected) <= set(record["input_candidate_ids"])
    assert set(selected).isdisjoint(remaining)
    assert len(remaining) == len(record["input_candidate_ids"]) - BUDGET_PER_ROUND
    assert len(labelled) == round_index * BUDGET_PER_ROUND
    assert labelled[-BUDGET_PER_ROUND:] == selected
    assert set(record["previous_selected_ids"]) <= set(labelled)
    assert REQUIRED_METRICS <= set(metrics)
    assert set(record["input_candidate_ids"]).isdisjoint(record["input_reference_ids"])
    assert record["input_reference_ids"] == unique_preserve(
        [*base_reference_ids, *record["input_labelled_ids"]]
    )
    assert manifest["input_checkpoint"] == str(record["input_checkpoint"])
    assert manifest["reference_count"] == len(record["input_reference_ids"])
    assert manifest["labelled_count_before"] == len(record["input_labelled_ids"])

    with np.load(directory / "candidate_proposals.npz", allow_pickle=True) as proposals:
        proposal_ids = [str(value) for value in proposals["image_ids"].tolist()]
    assert set(proposal_ids) == set(record["input_candidate_ids"])

    weights = manifest.get("acquisition_parameters", {}).get("weights", {})
    if variant_name == "full_p05":
        assert float(weights["coherence_power"]) == 0.5
    if variant_name == "full_p1":
        assert float(weights["coherence_power"]) == 1.0
    if strategy == "random":
        assert not (directory / "proposal_scores.csv").exists()
        assert not (directory / "image_scores.csv").exists()
        assert not (directory / "reference_proposals.npz").exists()
    else:
        assert (directory / "reference_proposals.npz").exists()
        assert (directory / "proposal_scores.csv").exists()
        assert (directory / "image_scores.csv").exists()
        assert manifest["reference_proposals_sha256"]
    if variant_name == "rarity_no_coherence":
        with (directory / "proposal_scores.csv").open(newline="", encoding="utf-8") as file:
            for row in csv.DictReader(file):
                assert np.isclose(float(row["rarity_bonus"]), float(row["rarity"]))

    metadata = {
        "variant": variant_name,
        "underlying_strategy": strategy,
        "round": int(round_index),
        "seed": int(SEED),
        "budget": int(BUDGET_PER_ROUND),
        "DAOWOD_commit": repo_commits["DAOWOD"],
        "PROB_commit": repo_commits["PROB"],
        "input_checkpoint": str(record["input_checkpoint"]),
        "coherence_power": float(VARIANTS[variant_name]["coherence_power"]),
        "candidate_ids_sha256": sha256_text(record["input_candidate_ids"]),
        "reference_ids_sha256": sha256_text(record["input_reference_ids"]),
        "labelled_before_sha256": sha256_text(record["input_labelled_ids"]),
        "evaluation_ids_sha256": evaluation_ids_sha256,
        "experiment_config_sha256": sha256_text(
            [json.dumps(experiment_config, sort_keys=True, default=json_default)]
        ),
        "resumed": bool(resumed),
    }
    existing_metadata_path = round_metadata_path(directory)
    if existing_metadata_path.exists():
        existing_metadata = json.loads(existing_metadata_path.read_text(encoding="utf-8"))
        comparable_existing = {**existing_metadata, "resumed": bool(resumed)}
        if comparable_existing != metadata:
            raise RuntimeError(f"Incompatible round metadata: {existing_metadata_path}")
    else:
        write_json(existing_metadata_path, metadata)
    return {
        "directory": directory,
        "manifest": manifest,
        "metrics": metrics,
        "selected": selected,
        "labelled": labelled,
        "remaining": remaining,
        "metadata": metadata,
    }


def validate_drive_round(variant_name, round_index, record):
    directory = round_dir(variant_name, round_index, root=DRIVE_RESULT_DIR)
    if not directory.exists():
        return None
    metadata_path = round_metadata_path(directory)
    if not metadata_path.exists():
        raise RuntimeError(f"Drive round lacks round_metadata.json: {directory}")
    data = validate_round_artifacts(variant_name, round_index, directory, record, resumed=True)
    expected_metadata = data["metadata"]
    saved_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if {**saved_metadata, "resumed": True} != expected_metadata:
        raise RuntimeError(f"Drive round metadata is incompatible: {metadata_path}")
    return data


def restore_completed_round(variant_name, round_index, record):
    source = round_dir(variant_name, round_index, root=DRIVE_RESULT_DIR)
    destination = round_dir(variant_name, round_index, root=LOCAL_RESULT_ROOT)
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(source, destination)
    data = validate_round_artifacts(variant_name, round_index, destination, record, resumed=True)
    write_json(round_metadata_path(destination), data["metadata"])
    return data


def atomic_copy_round_to_drive(variant_name, round_index, record):
    local_directory = round_dir(variant_name, round_index, root=LOCAL_RESULT_ROOT)
    drive_directory = round_dir(variant_name, round_index, root=DRIVE_RESULT_DIR)
    copying_directory = drive_directory.with_name(f"{drive_directory.name}.__copying__")
    if drive_directory.exists():
        raise RuntimeError(f"Completed Drive round already exists and will not be overwritten: {drive_directory}")
    if copying_directory.exists():
        shutil.rmtree(copying_directory)
    validate_round_artifacts(variant_name, round_index, local_directory, record, resumed=False)
    shutil.copytree(local_directory, copying_directory)
    validate_round_artifacts(variant_name, round_index, copying_directory, record, resumed=False)
    copying_directory.replace(drive_directory)
    validate_round_artifacts(variant_name, round_index, drive_directory, record, resumed=False)


def write_execution_status():
    data = {
        "experiment_name": EXPERIMENT_NAME,
        "status": STATUS,
        "round_status": {stage: STATUS[stage] for stage in ROUND_STAGES},
    }
    write_json(Path(LOCAL_RESULT_ROOT) / "execution_status.json", data)
    write_json(Path(DRIVE_RESULT_DIR) / "execution_status.json", data)


variant_states = {
    variant_name: {
        "current_checkpoint": TASK1_CHECKPOINT_FOR_PROB,
        "current_candidate_ids": list(candidate_ids),
        "cumulative_labelled_ids": list(initial_labelled_ids),
        "cumulative_reference_ids": list(base_reference_ids),
        "selected_history": [],
        "resumed_rounds": set(),
    }
    for variant_name in VARIANTS
}

initial_state_rows = []
for variant_name, state in variant_states.items():
    initial_state_rows.append(
        {
            "variant": variant_name,
            "checkpoint": state["current_checkpoint"],
            "candidate_count": len(state["current_candidate_ids"]),
            "reference_count": len(state["cumulative_reference_ids"]),
            "labelled_count": len(state["cumulative_labelled_ids"]),
            "evaluation_ids_sha256": evaluation_ids_sha256,
            "seed": SEED,
            "budget_per_round": BUDGET_PER_ROUND,
        }
    )
initial_fairness = pd.DataFrame(initial_state_rows)
display(initial_fairness)
assert initial_fairness.drop(columns=["variant"]).nunique().max() == 1

round_records = []
for variant_name, variant_config in VARIANTS.items():
    state = variant_states[variant_name]
    for round_index in range(1, ROUNDS + 1):
        stage = f"{variant_name} round {round_index}"
        input_checkpoint = state["current_checkpoint"]
        input_candidate_ids = list(state["current_candidate_ids"])
        input_labelled_ids = list(state["cumulative_labelled_ids"])
        input_reference_ids = unique_preserve([*base_reference_ids, *input_labelled_ids])
        previous_selected_ids = [
            image_id
            for selected_round in state["selected_history"]
            for image_id in selected_round
        ]
        record = {
            "variant": variant_name,
            "round": round_index,
            "input_checkpoint": str(input_checkpoint),
            "input_candidate_ids": input_candidate_ids,
            "input_labelled_ids": input_labelled_ids,
            "input_reference_ids": input_reference_ids,
            "previous_selected_ids": previous_selected_ids,
        }
        assert len(input_candidate_ids) == len(set(input_candidate_ids))
        assert len(input_reference_ids) == len(set(input_reference_ids))
        assert set(input_candidate_ids).isdisjoint(input_reference_ids)
        assert set(input_candidate_ids).isdisjoint(input_labelled_ids)
        assert len(input_candidate_ids) >= BUDGET_PER_ROUND
        assert sha256_text(evaluation_ids) == evaluation_ids_sha256

        drive_data = validate_drive_round(variant_name, round_index, record)
        if drive_data is not None:
            if not RESUME_COMPLETED_ROUNDS:
                raise RuntimeError(f"Completed Drive round exists but resume is disabled: {stage}")
            data = restore_completed_round(variant_name, round_index, record)
            state["resumed_rounds"].add(round_index)
            mark_restored(stage)
        else:
            local_directory = round_dir(variant_name, round_index, root=LOCAL_RESULT_ROOT)
            if local_directory.exists():
                local_manifest = local_directory / "round_manifest.json"
                if (
                    local_manifest.exists()
                    and json.loads(local_manifest.read_text(encoding="utf-8")).get("completed") is True
                ):
                    raise RuntimeError(
                        "Completed local round exists without a compatible Drive restore: "
                        f"{local_directory}"
                    )
                shutil.rmtree(local_directory)
            acquisition_config = build_acquisition_config(variant_config)
            result = run_active_round(
                adapter=adapter,
                checkpoint=input_checkpoint,
                candidate_ids=input_candidate_ids,
                reference_ids=input_reference_ids,
                labelled_ids=input_labelled_ids,
                output_dir=local_directory,
                strategy=variant_config["strategy"],
                budget=BUDGET_PER_ROUND,
                acquisition_config=acquisition_config,
                seed=SEED,
                round_index=round_index,
            )
            del result, acquisition_config
            data = validate_round_artifacts(variant_name, round_index, local_directory, record, resumed=False)
            atomic_copy_round_to_drive(variant_name, round_index, record)
            mark(stage)

        state["selected_history"].append(list(data["selected"]))
        state["current_checkpoint"] = str(round_dir(variant_name, round_index, root=LOCAL_RESULT_ROOT) / "checkpoint.pth")
        state["current_candidate_ids"] = list(data["remaining"])
        state["cumulative_labelled_ids"] = list(data["labelled"])
        state["cumulative_reference_ids"] = unique_preserve([*base_reference_ids, *data["labelled"]])
        round_records.append({**record, "resumed": round_index in state["resumed_rounds"]})
        write_execution_status()
        del data
        cleanup_runtime()


In [ ]:
METRIC_DIRECTIONS = {
    "known_mAP": "higher is better",
    "U_Recall": "higher is better",
    "WI": "lower is better",
    "A_OSE": "lower is better",
    "unknown_AP50": "higher is better",
    "previous_known_AP50": "higher is better",
}


def load_completed_round_for_summary(variant_name, round_index):
    directory = round_dir(variant_name, round_index, root=LOCAL_RESULT_ROOT)
    manifest = json.loads((directory / "round_manifest.json").read_text(encoding="utf-8"))
    metrics = json.loads((directory / "metrics.json").read_text(encoding="utf-8"))
    selected = read_ids(directory / "selected_ids.txt")
    labelled = read_ids(directory / "labelled_ids.txt")
    remaining = read_ids(directory / "remaining_pool_ids.txt")
    metadata = json.loads((directory / "round_metadata.json").read_text(encoding="utf-8"))
    return directory, manifest, metrics, selected, labelled, remaining, metadata


metric_rows = []
selected_ids_by_variant = {variant_name: {} for variant_name in VARIANTS}
for variant_name, variant_config in VARIANTS.items():
    for round_index in range(1, ROUNDS + 1):
        directory, manifest, metrics, selected, labelled, remaining, metadata = load_completed_round_for_summary(
            variant_name,
            round_index,
        )
        metric_rows.append(
            {
                "variant": variant_name,
                "underlying_strategy": variant_config["strategy"],
                "coherence_power": float(variant_config["coherence_power"]),
                "round": int(round_index),
                "round_budget": int(BUDGET_PER_ROUND),
                "cumulative_budget": int(round_index * BUDGET_PER_ROUND),
                "candidate_count_before": int(manifest["candidate_count_before"]),
                "candidate_count_after": int(manifest["candidate_count_after"]),
                "selected_images": int(len(selected)),
                "cumulative_labelled_images": int(len(labelled)),
                "reference_count": int(manifest["reference_count"]),
                "known_mAP": float(metrics["known_mAP"]),
                "U_Recall": float(metrics["U_Recall"]),
                "WI": float(metrics["WI"]),
                "A_OSE": int(metrics["A_OSE"]),
                "unknown_AP50": float(metrics["unknown_AP50"]),
                "previous_known_AP50": float(metrics["previous_known_AP50"]),
                "completed": bool(manifest["completed"]),
                "resumed": bool(metadata["resumed"]),
            }
        )
        selected_ids_by_variant[variant_name][f"round_{round_index:02d}"] = selected

metric_columns = [
    "variant",
    "underlying_strategy",
    "coherence_power",
    "round",
    "round_budget",
    "cumulative_budget",
    "candidate_count_before",
    "candidate_count_after",
    "selected_images",
    "cumulative_labelled_images",
    "reference_count",
    "known_mAP",
    "U_Recall",
    "WI",
    "A_OSE",
    "unknown_AP50",
    "previous_known_AP50",
    "completed",
    "resumed",
]
multiround_metrics = pd.DataFrame(metric_rows)[metric_columns]
if len(multiround_metrics) != len(VARIANTS) * ROUNDS:
    raise RuntimeError(f"Expected 12 metric rows, got {len(multiround_metrics)}.")
assert multiround_metrics["cumulative_budget"].tolist().count(10) == len(VARIANTS)
assert multiround_metrics["cumulative_budget"].tolist().count(20) == len(VARIANTS)
assert multiround_metrics["cumulative_budget"].tolist().count(30) == len(VARIANTS)

print("Metric interpretation directions:")
for metric, direction in METRIC_DIRECTIONS.items():
    print(f"- {metric}: {direction}")
display(multiround_metrics)

overlap_records = []
for round_index in range(1, ROUNDS + 1):
    sets = {
        variant_name: set(selected_ids_by_variant[variant_name][f"round_{round_index:02d}"])
        for variant_name in VARIANTS
    }
    for left in VARIANTS:
        for right in VARIANTS:
            overlap_records.append(
                {
                    "round": round_index,
                    "left_variant": left,
                    "right_variant": right,
                    "overlap_count": int(len(sets[left] & sets[right])),
                }
            )
    overlap_table = pd.DataFrame(
        {left: {right: int(len(sets[left] & sets[right])) for right in VARIANTS} for left in VARIANTS}
    ).loc[list(VARIANTS), list(VARIANTS)]
    overlap_table.index.name = f"round_{round_index:02d}"
    display(overlap_table)
multiround_overlap = pd.DataFrame(overlap_records)
final_round_comparison = multiround_metrics[multiround_metrics["round"] == ROUNDS].reset_index(drop=True)
learning_curve = multiround_metrics[
    [
        "variant",
        "underlying_strategy",
        "coherence_power",
        "cumulative_budget",
        "known_mAP",
        "U_Recall",
        "WI",
        "A_OSE",
        "unknown_AP50",
        "previous_known_AP50",
        "resumed",
    ]
].sort_values(["variant", "cumulative_budget"])
display(final_round_comparison)
display(learning_curve)

metrics_path = Path(LOCAL_RESULT_ROOT) / "multiround_metrics.csv"
selected_path = Path(LOCAL_RESULT_ROOT) / "multiround_selected_ids.json"
overlap_path = Path(LOCAL_RESULT_ROOT) / "multiround_overlap_by_round.csv"
summary_path = Path(LOCAL_RESULT_ROOT) / "experiment_summary.json"
learning_curve_path = Path(LOCAL_RESULT_ROOT) / "multiround_learning_curve.csv"
final_round_path = Path(LOCAL_RESULT_ROOT) / "multiround_final_round.csv"

multiround_metrics.to_csv(metrics_path, index=False)
selected_path.write_text(
    json.dumps(selected_ids_by_variant, indent=2, default=json_default) + "\n",
    encoding="utf-8",
)
multiround_overlap.to_csv(overlap_path, index=False)
learning_curve.to_csv(learning_curve_path, index=False)
final_round_comparison.to_csv(final_round_path, index=False)

experiment_summary = {
    "seed": int(SEED),
    "rounds": int(ROUNDS),
    "budget_per_round": int(BUDGET_PER_ROUND),
    "cumulative_budgets": [int(round_index * BUDGET_PER_ROUND) for round_index in range(1, ROUNDS + 1)],
    "initial_candidate_count": int(len(candidate_ids)),
    "base_reference_count": int(len(base_reference_ids)),
    "evaluation_count": int(len(evaluation_ids)),
    "imbalance_ratio": float(IMBALANCE_RATIO),
    "variants": VARIANTS,
    "effective_acquisition_parameters": effective_acquisition_parameters,
    "metric_directions": METRIC_DIRECTIONS,
    "metric_results": multiround_metrics.to_dict(orient="records"),
    "selected_ids_by_variant_and_round": selected_ids_by_variant,
    "overlap_records": multiround_overlap.to_dict(orient="records"),
    "learning_curve": learning_curve.to_dict(orient="records"),
    "final_round_comparison": final_round_comparison.to_dict(orient="records"),
    "DAOWOD_commit": str(repo_commits["DAOWOD"]),
    "PROB_commit": str(repo_commits["PROB"]),
    "benchmark_result": False,
}
write_json(summary_path, experiment_summary)

for artifact_path in (
    metrics_path,
    selected_path,
    overlap_path,
    learning_curve_path,
    final_round_path,
    summary_path,
    experiment_config_path,
    Path(LOCAL_RESULT_ROOT) / "execution_status.json",
):
    if not artifact_path.exists():
        raise FileNotFoundError(f"Missing final artifact: {artifact_path}")
saved_summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert saved_summary["benchmark_result"] is False
assert len(saved_summary["metric_results"]) == len(VARIANTS) * ROUNDS
for artifact_path in (
    metrics_path,
    selected_path,
    overlap_path,
    learning_curve_path,
    final_round_path,
    summary_path,
    experiment_config_path,
    Path(LOCAL_RESULT_ROOT) / "execution_status.json",
):
    shutil.copy2(artifact_path, Path(DRIVE_RESULT_DIR) / artifact_path.name)
for artifact_name in (
    "multiround_metrics.csv",
    "multiround_selected_ids.json",
    "multiround_overlap_by_round.csv",
    "multiround_learning_curve.csv",
    "multiround_final_round.csv",
    "experiment_summary.json",
    "experiment_config.json",
    "execution_status.json",
):
    if not (Path(DRIVE_RESULT_DIR) / artifact_name).exists():
        raise FileNotFoundError(f"Missing Drive summary artifact: {artifact_name}")
mark("final summary")
mark("Drive persistence")
write_execution_status()
cleanup_runtime()


In [ ]:
final_status = pd.DataFrame([{"stage": stage, "status": STATUS[stage]} for stage in STATUS_ORDER])
display(final_status)
rounds_done = all(STATUS[stage] in {"OK", "RESTORED"} for stage in ROUND_STAGES)
non_rounds_done = all(STATUS[stage] == "OK" for stage in STATUS_ORDER if stage not in ROUND_STAGES)
if not (rounds_done and non_rounds_done):
    raise RuntimeError("Contribution A multi-round pilot finished with one or more non-complete stages.")
print("Contribution A multi-round pilot completed successfully.")
